# UK Government Legislation Summary Tool

## Web Scraper

Below is a method for scraping readable policy text from https://www.legislation.gov.uk/ simply find the link to a compatable policy document and provide it to the tool.

An example policy you can use is https://www.legislation.gov.uk/ukpga/2025/18/contents

### Code

In [ ]:
pip install -r -q requirements.txt

In [ ]:
import feedparser
from bs4 import BeautifulSoup
import pandas as pd
import requests
import easygui
import logging
import re
import unittest

In [ ]:
class gov_uk_scraper:

    def input_url():
        # Prompt the user to enter the URL of the legislation's Table of Contents page
        myvar = easygui.enterbox("Please enter the URL to the Table of Contents page of the legislation you want to assess:", "UK Legislation NLP Comparison Tool")
        #if myvar is None:
        #    return None

        # Parse the RSS feed of the legislation's Table of Contents page
        # Includes error handling to catch any exceptions that may occur during parsing
        try:
            feed = feedparser.parse(myvar + "/data.feed")
            # Check if the feed has entries and get the first entry's link
            url = feed.entries[0].link
            return url
        except Exception:
            return gov_uk_scraper.input_error("notfound")
        
    def retry_input(option):
        if option == "Retry":
            return gov_uk_scraper.input_url()
        return None

    def input_error(error):
        def retry_input(option):
            if option == "Retry":
                return gov_uk_scraper.input_url()
            return None

        if error == "notfound":
            option = easygui.buttonbox("Unable to find a link to that legislation", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        elif error == "wrongsite":
            option = easygui.buttonbox("The URL provided does not appear to be a valid legislation.gov.uk page. Please check the URL and try again.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        elif error == "pdfpage":
            option = easygui.buttonbox("The legislation provided is only available as a PDF and unforunately incompatible with this tool.", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)
        elif error == "noemlink":
            option = easygui.buttonbox("The legislation provided does not appear to have an Explanatory Memorandum. No Comparison can be made", "UK Legislation NLP Comparison Tool", ["Retry", "Exit"])
            return retry_input(option)

    def get_legislation_text(url):
        if url.endswith(".pdf"):
            return gov_uk_scraper.input_error("pdfpage")
        else:
            response = requests.get(url)
            soup = BeautifulSoup(response.content, 'html.parser')
            #Final validation check to ensure website is legislation.gov.uk
            site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
            em_check = soup.find("li", id=["legEmLink","legEnLink"])
            if not site_check:
                return gov_uk_scraper.input_error("wrongsite")
            elif not em_check:
                return gov_uk_scraper.input_error("noemlink")
            else:
                legislation_text = soup.find("div", class_="LegSnippet")
                return legislation_text

    def prettify_text(content):
        lines = []
        for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
            # skip elements that just contain other elements we'll already visit —
            # otherwise you'd get duplicated text
            if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
                continue
            text = el.get_text(" ", strip=True)
            if text:
                lines.append(text)
        readable = "\n\n".join(lines)
        return readable

    def main():
        url = gov_uk_scraper.input_url()
        legislation_text = gov_uk_scraper.get_legislation_text(url)
        readable_text = gov_uk_scraper.prettify_text(legislation_text)
        print(readable_text)

In [ ]:
gov_uk_scraper.main()

### Unit Testing

In [ ]:
#Need to be completed still

# Early Testing sections

In [ ]:
def get_legislation_text(url):
    if url.endswith(".pdf"):
        return input_error("pdfpage")
    else:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        #Final validation check to ensure website is legislation.gov.uk
        site_check = "legislation.gov.uk" in soup.find("div", id="header").get_text()
        em_check = soup.find("li", id="legEmLink")
        if not site_check:
            return input_error("wrongsite")
        elif not em_check:
            return input_error("noemlink")
        else:
            legislation_text = soup.find("div", class_="LegSnippet")
        return legislation_text

In [ ]:
def prettify_text(content):
    lines = []
    for el in content.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "div"]):
        # skip elements that just contain other elements we'll already visit —
        # otherwise you'd get duplicated text
        if el.find(["p", "li", "td", "div", "h1", "h2", "h3", "h4"]):
            continue
        text = el.get_text(" ", strip=True)
        if text:
            lines.append(text)
    readable = "\n\n".join(lines)
    return readable

In [ ]:
def main():
    url = input_url()
    legislation_text = get_legislation_text(url)
    readable_text = prettify_text(legislation_text)
    print(readable_text)

In [ ]:
main()